# Notebook 00: dump discovery and provenance

This notebook opens two Geant4-DNA dumps written by `sim.exe`, reports what produced them (beam, seeds, Chemistry, chemistry mode, stage times, Geant4 runtime tag) and checks, on disk, which output files are present. It is the entry point of the notebook series: it reads no physics or chemistry results.

The default dumps are tiny, seeded runs (10 keV electrons, two events each). Everything below is a **workflow demonstration**, not a converged result. The IRT-only dump and the mesoscopic dump are two distinct simulations with different seeds, not one continuous trajectory.

Run the cells from top to bottom in a fresh kernel. To inspect other output, edit `IRT_DUMP` and `MESO_DUMP` in the next cell.

## 1. Configuration and input checks

The defaults point at ignored regenerated data under `DEV_DIR`. Use unprefixed dump directories that contain their own `Manifest.json`: not a results-index directory and not a filename-prefixed dump. The cell fails early, with regeneration guidance, if G4Utils lacks the readers used later in the series (`Dump.track_lengths` and friends) or if a dump is missing.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Markdown, display

import g4utils
from g4utils.DnaChem import Dump, Simulation

dev_dir = os.environ.get('DEV_DIR')
if not dev_dir:
    raise RuntimeError(
        'DEV_DIR is unset. Run ~/dotfiles/scripts/windows/setup-dev-env.ps1 and restart the shell before using this notebook.'
    )
DEV_DIR = Path(dev_dir)

DEFAULT_RUN_ROOT = (
    DEV_DIR / 'GEANT4' / 'SIM' / 'dnachem-min'
    / '.scratch' / 'tests'
    / '2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the'
)

# Edit these two paths for a different pair of dumps.
IRT_DUMP = DEFAULT_RUN_ROOT / 'irt'
MESO_DUMP = DEFAULT_RUN_ROOT / 'meso-spatial'

README_HINT = (
    'See notebooks/README.md ("Reproduce the runs") to regenerate the demonstration data, '
    'and ("Python and Jupyter") to install a G4Utils checkout that includes the dump readers.'
)

# 1. The installed G4Utils must expose the readers used by this series.
REQUIRED_DUMP_API = [
    'from_manifest', 'track_lengths', 'physical_interactions', 'prechemical_records',
    'meso_species', 'has_meso_species', 'has_meso', 'particle_g_estimates',
]
missing_api = [name for name in REQUIRED_DUMP_API if not hasattr(Dump, name)]
if missing_api:
    raise ImportError(
        f'The G4Utils at {Path(g4utils.__file__).parent} lacks Dump.{missing_api}. '
        f'Install a revision that includes them (editable install of the G4Utils checkout, '
        f'then restart the kernel). {README_HINT}'
    )

# 2. The dumps and their manifests must exist.
DUMPS = {'IRT': IRT_DUMP, 'mesoscopic': MESO_DUMP}
for label, path in DUMPS.items():
    setting = 'IRT_DUMP' if label == 'IRT' else 'MESO_DUMP'
    if not path.is_dir():
        raise FileNotFoundError(
            f'{label} dump directory is unavailable: {path}. Edit {setting} or regenerate it. {README_HINT}'
        )
    manifest = path / 'Manifest.json'
    if not manifest.is_file() or manifest.stat().st_size == 0:
        raise FileNotFoundError(
            f'{label} dump has no usable Manifest.json: {manifest}. Point {setting} at an unprefixed dump '
            f'directory (not the results index). {README_HINT}'
        )

print(f'Kernel      : {sys.executable}')
print(f'G4Utils     : {Path(g4utils.__file__).parent}')
for label, path in DUMPS.items():
    print(f'{label:<12}: {path}')

Kernel      : C:\Users\rtonneau\AppData\Local\anaconda3\python.exe
G4Utils     : C:\DEV\python\G4Utils\src\g4utils
IRT         : C:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\irt
mesoscopic  : C:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\meso-spatial


## 2. Discover the dumps

`Dump.from_manifest(path)` selects one manifest explicitly. `Simulation(root)` discovers every dump under a directory and offers a one-row-per-dump summary table. Here `Simulation` is pointed at each selected dump directory only, so unrelated sibling output is not picked up.

In [2]:
dumps = {label: Dump.from_manifest(path / 'Manifest.json') for label, path in DUMPS.items()}

for label, dump in dumps.items():
    resolved = dump.path.resolve()
    assert resolved == DUMPS[label].resolve(), f'{label}: manifest resolved to {resolved}, expected {DUMPS[label]}'
    print(f'{label:<11} dump name={dump.name!r}  prefix={dump.prefix!r}  labels={dict(dump.labels)}')

display(Markdown('### `Simulation(...).table()` for each selected dump'))
for label, path in DUMPS.items():
    simulation = Simulation(path)
    table = simulation.table()
    assert len(table) == 1, f'{label}: expected exactly one dump under {path}, found {len(table)}'
    display(table)

IRT         dump name='irt'  prefix=''  labels={}
mesoscopic  dump name='meso-spatial'  prefix=''  labels={}


### `Simulation(...).table()` for each selected dump

,dump,chemistry,pH,totalEvents,totalEnergyDeposit_eV,particle,beamEnergy_keV,dump_key,manifest_path,prefix,chemistryModel,handOverTime_ns,chemistryEndTime_ns,voxelSize_nm,mesoPixels,mesoTimesPerDecade,mesoSpatialOutput,threads,wallTime_s
0,irt,PureWater,7,2,20000,e-,10,Manifest.json,C:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...,,IRT_syn,None,1000,None,None,None,False,1,4.7937


,dump,chemistry,pH,totalEvents,totalEnergyDeposit_eV,particle,beamEnergy_keV,dump_key,manifest_path,prefix,chemistryModel,handOverTime_ns,chemistryEndTime_ns,voxelSize_nm,mesoPixels,mesoTimesPerDecade,mesoSpatialOutput,threads,wallTime_s
0,meso-spatial,PureWater,7,2,20000,e-,10,Manifest.json,C:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...,,IRT_syn+mesoscopic,5,1000000,6.25,65536,10,True,1,19.003541


## 3. Provenance

Everything here comes from `Manifest.json` through the G4Utils `Manifest` object. The manifest records the **first** seed of the pair passed to `/random/setSeeds`; the second seed is only in the macro. Stage times are `n/a` when a stage did not run: an IRT-only run has no hand-over, and the particle-stage species files then span the whole run, up to the chemistry end time. Whether the mesoscopic stage ran is read from the `mesoEnabled` flag, falling back to the exact model string `IRT_syn+mesoscopic` only for historical manifests that lack it.

In [3]:
def meso_stage_ran(manifest) -> bool:
    if manifest.mesoEnabled is not None:
        return bool(manifest.mesoEnabled)
    return manifest.chemistryModel == 'IRT_syn+mesoscopic'


def fmt_ns(value) -> str:
    return 'n/a' if value is None else f'{value:,.10g} ns'


def provenance_row(label: str, dump: Dump) -> dict:
    m = dump.manifest
    meso = meso_stage_ran(m)
    return {
        'dump': label,
        'macro': m.macro,
        'Geant4 runtime tag': m.geant4Version,
        'Chemistry': m.chemistry,
        'time-step model': m.raw.get('timeStepModel'),
        'chemistry mode': m.chemistryModel,
        'mesoscopic stage': 'on' if meso else 'off',
        'hand-over time': fmt_ns(m.handOverTime_ns if meso else None),
        'chemistry end time': fmt_ns(m.chemistryEndTime_ns),
        'run mode / threads': f'{m.runMode} / {m.threads}',
        'events': m.totalEvents,
        'deposited energy (eV)': m.totalEnergyDeposit_eV,
        'pH': m.pH,
        'added scavengers': ', '.join(str(s) for s in m.scavengers) or 'none',
        'half box (um)': m.halfBox_um,
    }


def beam_rows(label: str, dump: Dump) -> pd.DataFrame:
    table = dump.manifest.runs_table()
    table.insert(0, 'dump', label)
    return table


provenance = pd.DataFrame([provenance_row(label, dump) for label, dump in dumps.items()]).set_index('dump').T
display(Markdown('### Dump provenance'))
display(provenance)

display(Markdown('### Beam, seed and per-run records'))
display(pd.concat([beam_rows(label, dump) for label, dump in dumps.items()], ignore_index=True))

# Consistency: per-run numbers must add up to the manifest totals.
for label, dump in dumps.items():
    m = dump.manifest
    assert sum(r.events for r in m.runs) == m.totalEvents, f'{label}: run events do not sum to totalEvents'
    assert abs(sum(r.energyDeposit_eV for r in m.runs) - m.totalEnergyDeposit_eV) <= 1e-6 * max(1.0, m.totalEnergyDeposit_eV), \
        f'{label}: run energy deposits do not sum to totalEnergyDeposit_eV'
    assert m.totalEvents > 0 and m.chemistryEndTime_ns, f'{label}: empty dump'
    if meso_stage_ran(m):
        assert m.handOverTime_ns is not None and m.handOverTime_ns < m.chemistryEndTime_ns
print('Provenance consistency checks: PASS')

Provenance consistency checks: PASS


### Dump provenance

dump,IRT,mesoscopic
macro,..\notebooks\demo_irt.in,..\notebooks\demo_meso_spatial.in
Geant4 runtime tag,geant4-11-04-patch-01,geant4-11-04-patch-01
Chemistry,PureWater,PureWater
time-step model,IRT_syn,IRT_syn
chemistry mode,IRT_syn,IRT_syn+mesoscopic
mesoscopic stage,off,on
hand-over time,n/a,5 ns
chemistry end time,"1,000 ns","1,000,000 ns"
run mode / threads,Serial / 1,Serial / 1
events,2,2


### Beam, seed and per-run records

,dump,run,events,particle,beamEnergy_keV,position_um,direction,energyDeposit_eV,seed,wallTime_s
0,IRT,0,2,e-,10,"(0, 0, 0)","(0, 0, 1)",20000,12345,4.793700
1,mesoscopic,0,2,e-,10,"(0, 0, 0)","(0, 0, 1)",20000,23456,19.003541


## 4. File inventory checked on disk

`Manifest.json` lists the data files of a dump, but a listed file can still be missing, empty or replaced. The inventory therefore compares three things per dump: files the manifest declares, files actually on disk, and files the chemistry mode should have produced. A declared file that is absent or empty raises an error, because the readers in later notebooks select data from the manifest inventory and do not substitute another file.

Expected by mode: the core particle-stage and physical-stage files are always written; `SpeciesMeso.*` only when the mesoscopic stage ran; `SpeciesMesoSpatial.h5` only with `/chem/meso/spatialOutput true`; one `PreChemical_run<R>_event<E>.txt` per event when pre-chemical output is on. `Dump.has_meso` refers to the **spatial HDF5 file**, whereas `Dump.has_meso_species` refers to the aggregate `SpeciesMeso.csv`.

In [4]:
CORE_FILES = [
    'Species.Txt', 'Species_nt_species.csv', 'Reactions.Txt', 'Reactions_nt_reactions.csv',
    'ReactionsMetadata.csv', 'PhysicsInteractions.Txt', 'PhysicsInteractions.csv',
    'TrackLengths.Txt', 'TrackLengths.csv',
]
MESO_FILES = ['SpeciesMeso.Txt', 'SpeciesMeso.csv']
SPATIAL_FILES = ['SpeciesMesoSpatial.h5']

DESCRIPTION = {
    'Manifest.json': 'provenance and file inventory',
    'Species.Txt': 'particle-stage species (text)', 'Species_nt_species.csv': 'particle-stage species ntuple',
    'Reactions.Txt': 'particle-stage reactions (text)', 'Reactions_nt_reactions.csv': 'particle-stage reaction ntuple',
    'ReactionsMetadata.csv': 'reaction id/label table',
    'PhysicsInteractions.Txt': 'physical interactions (text)', 'PhysicsInteractions.csv': 'physical interaction counts',
    'TrackLengths.Txt': 'per-event track lengths (text)', 'TrackLengths.csv': 'per-event track lengths',
    'SpeciesMeso.Txt': 'mesoscopic mean count per event', 'SpeciesMeso.csv': 'mesoscopic summed counts',
    'SpeciesMesoSpatial.h5': 'mesoscopic per-event snapshots',
}


def description(name: str) -> str:
    if name.startswith('PreChemical_run'):
        return 'pre-chemical records of one event'
    return DESCRIPTION.get(name, 'not documented in this notebook')


def expected_files(dump: Dump) -> list[str]:
    m = dump.manifest
    expected = ['Manifest.json', *CORE_FILES]
    if meso_stage_ran(m):
        expected += MESO_FILES
    if m.mesoSpatialOutput:
        expected += SPATIAL_FILES
    return expected


def inventory(label: str, dump: Dump) -> pd.DataFrame:
    on_disk = {p.name: p for p in dump.path.iterdir() if p.is_file()}
    declared = set(dump.files or ())
    expected = set(expected_files(dump))
    rows = []
    for name in sorted(declared | set(on_disk) | expected):
        path = on_disk.get(name)
        size = None if path is None else path.stat().st_size
        rows.append({
            'dump': label,
            'file': name,
            'role': description(name),
            'declared': name in declared or name == 'Manifest.json',
            'expected for mode': name in expected,
            'on disk': path is not None,
            'size (KiB)': None if size is None else round(size / 1024, 1),
            'non-empty': bool(size),
        })
    return pd.DataFrame(rows)


tables = {label: inventory(label, dump) for label, dump in dumps.items()}
file_inventory = pd.concat(tables.values(), ignore_index=True)

# Declared or expected files must be present and non-empty.
problems = file_inventory[(file_inventory['declared'] | file_inventory['expected for mode'])
                          & ~file_inventory['non-empty']]
if not problems.empty:
    raise FileNotFoundError(
        'Declared or expected files are missing or empty:\n'
        + problems[['dump', 'file', 'declared', 'expected for mode', 'on disk']].to_string(index=False)
        + '\nRegenerate the dump with those outputs enabled (notebooks/README.md) or edit IRT_DUMP / MESO_DUMP.'
    )

# Pre-chemical files: one per event if present at all.
for label, dump in dumps.items():
    n_prechem = int(tables[label]['file'].str.startswith('PreChemical_run').sum())
    print(f'{label}: {len(tables[label])} files inventoried, {n_prechem} pre-chemical event file(s), '
          f'{dump.manifest.totalEvents} events')

summary = pd.DataFrame({
    label: {
        'files on disk': int(t['on disk'].sum()),
        'declared in manifest': int(t['declared'].sum()),
        'undeclared on disk': ', '.join(t.loc[t['on disk'] & ~t['declared'], 'file']) or 'none',
        'total size (KiB)': round(t['size (KiB)'].sum(), 1),
        'Dump.has_meso_species (SpeciesMeso.csv)': dumps[label].has_meso_species,
        'Dump.has_meso (spatial HDF5)': dumps[label].has_meso,
    }
    for label, t in tables.items()
}).rename_axis('item')
display(Markdown('### Inventory summary'))
display(summary)
display(Markdown('### Per-file inventory'))
display(file_inventory)
print('File inventory checks: PASS')

IRT: 12 files inventoried, 2 pre-chemical event file(s), 2 events
mesoscopic: 15 files inventoried, 2 pre-chemical event file(s), 2 events
File inventory checks: PASS


### Inventory summary

,IRT,mesoscopic
item,,
files on disk,12,15
declared in manifest,12,15
undeclared on disk,none,none
total size (KiB),330.2,6994.2
Dump.has_meso_species (SpeciesMeso.csv),False,True
Dump.has_meso (spatial HDF5),False,True


### Per-file inventory

,dump,file,role,declared,expected for mode,on disk,size (KiB),non-empty
0,IRT,Manifest.json,provenance and file inventory,True,True,True,2.5,True
1,IRT,PhysicsInteractions.Txt,physical interactions (text),True,True,True,0.2,True
2,IRT,PhysicsInteractions.csv,physical interaction counts,True,True,True,0.2,True
3,IRT,PreChemical_run0_event0.txt,pre-chemical records of one event,True,False,True,143.2,True
4,IRT,PreChemical_run0_event1.txt,pre-chemical records of one event,True,False,True,142.0,True
5,IRT,Reactions.Txt,particle-stage reactions (text),True,True,True,4.0,True
6,IRT,ReactionsMetadata.csv,reaction id/label table,True,True,True,0.8,True
7,IRT,Reactions_nt_reactions.csv,particle-stage reaction ntuple,True,True,True,1.1,True
8,IRT,Species.Txt,particle-stage species (text),True,True,True,8.8,True
9,IRT,Species_nt_species.csv,particle-stage species ntuple,True,True,True,26.8,True


## 5. What this establishes

- Both dumps were opened explicitly from their own `Manifest.json`, and each is a single, unprefixed dump.
- Beam, first seed, Chemistry, chemistry mode, stage times, event count, deposited energy and the Geant4 runtime tag are read from the manifests, not assumed. The runtime tag identifies the Geant4 build that wrote the dump; it does not identify the `dnachem-min` source revision.
- Every declared file exists and is non-empty on disk. The mesoscopic dump additionally provides `SpeciesMeso.*` and the spatial HDF5 file, and the IRT-only dump does not.

The next notebooks read the physical, track-length, pre-chemical, species and reaction files from these dumps. With two events per dump, all later numbers are workflow demonstrations rather than converged radiolysis results.